# Missing Data Imputation

using MICE (Multiple Imputation by Chained Equations)


TODO:

You'll have to implement the following new classes:

LKJCholesky distribution (for prior for the below p
LKJCholeskyCorrParameter parameter (covariance or correlation?)
    or, maybe just do one which generates the whole Covariance matrix?
MultivariateNormalCholesky distribution
    same as multivariate dist but takes a TriL matrix (vector?) insetad of cov matrix
    ... but should probs be matrix so you don't have to worry about "how do we index this to get the matrix, which might differ between backends) instead of
    OR, add this as alt param to MultivariateNormal? maybe that's better.  see what pytorch + TFP do


Feel like you'll have to override the log_likelihood method for this?




In [ ]:
class MultipleImputationByChainedEquations(pf.Model):

    def __init__(self, d: int, n_missing: int):
        self.d = d
        self.cholesky_corr = pf.LKJCholeskyCorrParameter(
            d=d,
            prior=pf.LKJCholeskyCorr(d=d, eta=2),
        )  # correlation matrix
        self.std = pf.ScaleParameter(shape=d)  # standard deviations for each dimension
        self.mu = pf.Parameter(shape=d)
        self.z = pf.Parameter(shape=n_missing)  # latent parameters for missing data

    def __call__(self):
        L = self.cholesky_corr()
        cov = L @ tf.linalg.diag(self.std()) @ L.T
        return pf.MultivariateNormal(loc=self.mu(), cov=cov)

    def log_likelihood(self, missing_ix, y):
        """This method treats what is usually the x parameter as the missing data 
        index matrix, and y as the data matrix.  The missing data matrix is the same
        size as y, but when an element has a >=0 value, that value is the ID of the
        missing data element)

        E.g.:

        y = [
            [0, NaN, 2],
            [3, 4, NaN],
            [NaN, 7, NaN],
        ]

        missing_ix = [
            [-1, 0, -1],
            [-1, -1, 1],
            [2, -1, 3],
        ]
        """
        # For missing data, sample from imputed data latent posteriors
        missing_mask = missing_ix >= 0
        safe_unobs_ix = tf.where(missing_mask, missing_ix, 0)
        unobs_y = self.z[safe_unobs_ix]

        # Combine observed and imputed data
        combined_x = tf.where(missing_mask, unobs_y, y)

        # Calculate log likelihood of observed + imputed data
        # w.r.t. underlying multivariate normal model
        log_likelihoods = self().log_prob(combined_x)
        return O.sum(log_likelihoods, axis=None)


y = # some data matrix with NaNs where there are missing data

# Generate a missing data index matrix
missing_ix = np.full_like(y, -1)
missing_ix[np.isnan(y)] = np.arange(np.isnan(y).sum())

# Create and fit the MICE model
mice_model = MultipleImputationByChainedEquations(d=y.shape[1], n_missing=np.isnan(y).sum())
mice_model.fit(missing_ix, y)